# Experiment: ADC2023 FMPE Training Runbook

Objective:
- Prepare the ADC2023 five-gas dataset and train the current non-quantum FMPE / flow-matching model.
- Launch training in a persistent `tmux` session, tail the logs, and inspect periodic RMSE / mRMSE outputs.

Notes:
- This notebook is aligned to the current repo implementation in `models/sbi_ariel_adc2023`.
- It can reuse the active Vast run at `/workspace/hack4sages/local_runs/adc2023_20260311_203436_retry2` if that directory exists.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return candidate
        nested_root = candidate / "ariel" / "models" / "notebooks"
        if (nested_root / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return nested_root
    raise FileNotFoundError(
        "Could not locate the Ariel ADC2023 notebook project root from the current working directory."
    )


def find_data_root(project_root: Path) -> Path:
    candidates = [
        Path("/workspace/hack4sages/data/full-ariel"),
        Path("/workspace/hack4sages/ariel"),
        project_root / "data" / "full-ariel",
        project_root.parent.parent,
    ]
    for candidate in candidates:
        if (candidate / "TrainingData").exists() and (candidate / "TestData").exists():
            return candidate
    return candidates[-1]


def stream_bash(script: str, *, cwd: Path | None = None, env: dict[str, str] | None = None, check: bool = True) -> int:
    proc = subprocess.Popen(
        ["bash", "-lc", script],
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    return proc.returncode


def capture_bash(script: str, *, cwd: Path | None = None, env: dict[str, str] | None = None, check: bool = True) -> str:
    completed = subprocess.run(
        ["bash", "-lc", script],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed.stdout


def tail_text(path: Path, *, lines: int = 40) -> str:
    if not path.exists():
        return f"{path} does not exist yet."
    quoted = shlex.quote(str(path))
    return capture_bash(f"tail -n {int(lines)} {quoted}", check=False)


def latest_jsonl_record(path: Path) -> dict | None:
    if not path.exists():
        return None
    lines = [line for line in path.read_text().splitlines() if line.strip()]
    if not lines:
        return None
    return json.loads(lines[-1])


## Plan

- Bootstrap a dedicated training virtual environment that matches the Vast run.
- Verify the raw ADC2023 files and prepare the normalized five-gas FMPE dataset.
- Reuse the active Vast run if it exists, otherwise launch a fresh `tmux`-backed training session.
- Monitor `train.log` and `validation_rmse.jsonl`, then run the evaluation/export step when training is ready.


In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEFAULT_LOCAL_DATA_ROOT = find_data_root(PROJECT_ROOT)
DEFAULT_REMOTE_DATA_ROOT = Path("/workspace/hack4sages/data/full-ariel")
DATA_ROOT = DEFAULT_REMOTE_DATA_ROOT if DEFAULT_REMOTE_DATA_ROOT.exists() else DEFAULT_LOCAL_DATA_ROOT
PREPARED_DATA = PROJECT_ROOT / "data" / "generated-data" / "adc2023_fivegas_fmpe_prepared"
SETTINGS_FILE = PROJECT_ROOT / "models" / "sbi_ariel_adc2023" / "settings" / "adc2023_rtx4090.yaml"
IS_MACOS = platform.system() == "Darwin"
IS_LINUX = platform.system() == "Linux"
IS_REMOTE_WORKSPACE = str(PROJECT_ROOT).startswith("/workspace/")
REQUIREMENTS_FILE = PROJECT_ROOT / "models" / "sbi_ariel_crossgen" / (
    "requirements-mac.txt" if IS_MACOS else "requirements-vast.txt"
)
LAUNCHER = PROJECT_ROOT / "models" / "run_sbi_ariel_adc2023_ubuntu4090.sh"
VENV_DIR = PROJECT_ROOT / ".venv-sbi-adc2023"
TORCH_INSTALL_CMD = (
    "python -m pip install torch"
    if IS_MACOS
    else "python -m pip install --index-url https://download.pytorch.org/whl/cu121 torch"
)
DINGO_INSTALL_CMD = (
    "echo 'Skipping dingo-gw install on macOS preflight.'"
    if IS_MACOS
    else "python -m pip install --no-deps dingo-gw==0.8.3"
)

USE_EXISTING_ACTIVE_RUN = IS_REMOTE_WORKSPACE
KNOWN_ACTIVE_RUN = Path("/workspace/hack4sages/local_runs/adc2023_20260311_203436_retry2")
RUN_NAME = os.environ.get("ADC2023_RUN_NAME") or f"adc2023_fmpe_notebook_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR = KNOWN_ACTIVE_RUN if USE_EXISTING_ACTIVE_RUN and KNOWN_ACTIVE_RUN.exists() else PROJECT_ROOT / "local_runs" / RUN_NAME
TMUX_SESSION = os.environ.get("ADC2023_TMUX_SESSION") or RUN_DIR.name

TRAIN_LOG = RUN_DIR / "train.log"
VALIDATION_RMSE = RUN_DIR / "validation_rmse.jsonl"
VALIDATION_METRICS = RUN_DIR / "validation_regression_metrics.json"
HOLDOUT_METRICS = RUN_DIR / "holdout_regression_metrics.json"
TEST_PREDICTIONS = RUN_DIR / "testdata_predictions.csv"

config = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "prepared_data": str(PREPARED_DATA),
    "settings_file": str(SETTINGS_FILE),
    "requirements_file": str(REQUIREMENTS_FILE),
    "launcher": str(LAUNCHER),
    "venv_dir": str(VENV_DIR),
    "run_dir": str(RUN_DIR),
    "tmux_session": TMUX_SESSION,
    "platform": platform.platform(),
    "is_macos": IS_MACOS,
    "is_remote_workspace": IS_REMOTE_WORKSPACE,
    "torch_install_cmd": TORCH_INSTALL_CMD,
}
config


In [ ]:
required_files = [
    DATA_ROOT / "TrainingData" / "AuxillaryTable.csv",
    DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv",
    DATA_ROOT / "TrainingData" / "SpectralData.hdf5",
    DATA_ROOT / "TestData" / "AuxillaryTable.csv",
    DATA_ROOT / "TestData" / "SpectralData.hdf5",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing ADC2023 files:\n" + "\n".join(missing))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Platform: {platform.platform()}")
print(f"Remote workspace: {IS_REMOTE_WORKSPACE}")
print("\nGPU probe:")
capture_bash("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", cwd=PROJECT_ROOT, check=False)
if IS_MACOS:
    print("\nLocal macOS preflight detected. Environment bootstrap and dataset preparation can run locally, but the bundled launch script and default settings target Linux + CUDA.")
print("\nRequired files are present.")


In [ ]:
activate_path = VENV_DIR / "bin" / "activate"
bootstrap_script = f"""
set -euo pipefail
python3 -m venv {shlex.quote(str(VENV_DIR))}
source {shlex.quote(str(activate_path))}
python -m pip install --upgrade pip
{TORCH_INSTALL_CMD}
{DINGO_INSTALL_CMD}
python -m pip install -r {shlex.quote(str(REQUIREMENTS_FILE))}
"""
stream_bash(bootstrap_script, cwd=PROJECT_ROOT)


In [ ]:
prepare_manifest = PREPARED_DATA / "manifest.json"
prepare_script = f"""
set -euo pipefail
source {shlex.quote(str(activate_path))}
if [[ -f {shlex.quote(str(prepare_manifest))} ]]; then
  echo 'Prepared dataset already exists.'
else
  python -m models.sbi_ariel_adc2023.prepare_dataset \
    --data-root {shlex.quote(str(DATA_ROOT))} \
    --output {shlex.quote(str(PREPARED_DATA))} \
    --overwrite
fi
"""
stream_bash(prepare_script, cwd=PROJECT_ROOT)


In [ ]:
manifest = json.loads((PREPARED_DATA / "manifest.json").read_text())
manifest

print("\nSettings file preview:\n")
capture_bash(f"sed -n '1,220p' {shlex.quote(str(SETTINGS_FILE))}", cwd=PROJECT_ROOT, check=False)


In [ ]:
if IS_MACOS and not IS_REMOTE_WORKSPACE:
    print("Local macOS detected. Skipping tmux launch because the bundled launcher and default YAML target Linux + CUDA.")
    print("Use the setup and dataset-preparation cells locally, or run this launch cell from a Linux GPU environment.")
else:
    launch_script = f"""
set -euo pipefail
mkdir -p {shlex.quote(str(RUN_DIR))}
if ! command -v tmux >/dev/null 2>&1; then
  if command -v apt-get >/dev/null 2>&1; then
    apt-get update && apt-get install -y tmux
  else
    echo 'tmux is required for persistent notebook-driven training.'
    exit 1
  fi
fi
if tmux has-session -t {shlex.quote(TMUX_SESSION)} 2>/dev/null; then
  echo 'tmux session already exists: {TMUX_SESSION}'
else
  tmux new-session -d -s {shlex.quote(TMUX_SESSION)} \
    \"cd {shlex.quote(str(PROJECT_ROOT))} && export DATA_ROOT={shlex.quote(str(DATA_ROOT))} && export PREPARED_DATA={shlex.quote(str(PREPARED_DATA))} && export RUN_DIR={shlex.quote(str(RUN_DIR))} && export SETTINGS_FILE={shlex.quote(str(SETTINGS_FILE))} && export VENV_DIR={shlex.quote(str(VENV_DIR))} && bash {shlex.quote(str(LAUNCHER))}\"
fi
tmux ls
"""
    stream_bash(launch_script, cwd=PROJECT_ROOT)
    print(f"Run directory: {RUN_DIR}")


In [ ]:
from IPython.display import clear_output


def show_run_status() -> None:
    clear_output(wait=True)
    print(f"Run directory: {RUN_DIR}")
    print(f"tmux session: {TMUX_SESSION}")
    capture_bash(f"tmux has-session -t {shlex.quote(TMUX_SESSION)} 2>/dev/null && echo 'tmux session is active' || echo 'tmux session is not active'", cwd=PROJECT_ROOT, check=False)
    print("\nRecent train.log:\n")
    print(tail_text(TRAIN_LOG, lines=40))
    latest_rmse = latest_jsonl_record(VALIDATION_RMSE)
    if latest_rmse is None:
        print("validation_rmse.jsonl has not been written yet.")
    else:
        print("Latest validation RMSE monitor record:\n")
        print(json.dumps(latest_rmse, indent=2, sort_keys=True))


show_run_status()

# Optional live watch.
# Set WATCH_SECONDS > 0 to refresh this cell repeatedly.
WATCH_SECONDS = 0
SLEEP_SECONDS = 5
if WATCH_SECONDS > 0:
    deadline = time.time() + WATCH_SECONDS
    while time.time() < deadline:
        show_run_status()
        time.sleep(SLEEP_SECONDS)


In [ ]:
evaluation_script = f"""
set -euo pipefail
source {shlex.quote(str(activate_path))}
python -u -m models.sbi_ariel_adc2023.evaluate \
  --run-dir {shlex.quote(str(RUN_DIR))} \
  --prepared-data {shlex.quote(str(PREPARED_DATA))}
"""

# Uncomment when you want to export holdout metrics and test predictions.
# stream_bash(evaluation_script, cwd=PROJECT_ROOT)

for path in [VALIDATION_METRICS, HOLDOUT_METRICS, TEST_PREDICTIONS]:
    print(path, path.exists())
    if path.exists() and path.suffix == ".json":
        print(path.read_text()[:2000])
    elif path.exists() and path.suffix == ".csv":
        capture_bash(f"head -n 5 {shlex.quote(str(path))}", cwd=PROJECT_ROOT, check=False)


## Next steps

- To reuse the live Vast run, leave `USE_EXISTING_ACTIVE_RUN = True` and keep monitoring `train.log`.
- To start a fresh run, set `USE_EXISTING_ACTIVE_RUN = False`, rerun the config cell, then rerun the launch cell.
- After training writes `best_model_by_mrmse.pt`, run the evaluation cell to export holdout metrics and `testdata_predictions.csv`.
